# UTS Pembelajaran Mesin — Klasifikasi Status Mutu Air Sungai DKI Jakarta (KNN)

**Dataset:** Data Kualitas Air Sungai 2022 — Satu Data Jakarta (Dinas Lingkungan Hidup DKI Jakarta)
https://satudata.jakarta.go.id/open-data/detail/data-kualitas-air-sungai-2022

**Masalah:** memprediksi *status pencemaran* (`CEMAR RINGAN / SEDANG / BERAT`, dihitung dengan Indeks Pencemaran/IP)
sebuah titik sampel sungai dari hasil pengukuran parameter laboratorium, sehingga titik yang berisiko dapat
disaring lebih cepat sebelum perhitungan IP lengkap.

**Algoritma:** K-Nearest Neighbors (KNN) — sama dengan perhitungan manual pada Soal 2.

Alur: (1) muat & rapikan data → (2) EDA → (3) preprocessing → (4) model KNN → (5) evaluasi → (6) verifikasi hitungan manual.

In [1]:
# 202431133_AzkaNazalla
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
FILE = "data-kualitas-air-sungai-2022-komponen-data.xls"

## 1. Memuat data

In [2]:
raw = pd.read_html(FILE, thousands=None, converters={c: str for c in range(14)})[0]
raw["parameter"] = raw["parameter"].str.strip()
raw["jenis_parameter"] = raw["jenis_parameter"].fillna("IP")
print("Bentuk data mentah (format panjang):", raw.shape)
raw.head(8)

Bentuk data mentah (format panjang): (14512, 14)


,periode_data,tahun,periode,bulan_sampling,titik_sampel,sungai,alamat,lintang,bujur,jenis_parameter,parameter,satuan,baku_mutu,hasil_pengukuran
0,2022,2022,1,202203,ANK-1,ANGKE,JL. DURI KOSAMBI CILEDUG,"-6,175882","106,728695",FISIKA,ZAT PADAT TERLARUT TDS,MG L,1000,170
1,2022,2022,1,202203,ANK-1,ANGKE,JL. DURI KOSAMBI CILEDUG,"-6,175882","106,728695",FISIKA,ZAT PADAT TERSUSPENSI TSS,MG L,50,19
2,2022,2022,1,202203,ANK-1,ANGKE,JL. DURI KOSAMBI CILEDUG,"-6,175882","106,728695",KIMIA,PH,-,6 - 9,"7,57"
3,2022,2022,1,202203,ANK-1,ANGKE,JL. DURI KOSAMBI CILEDUG,"-6,175882","106,728695",KIMIA,BOD,MG L,3,"15,35"
4,2022,2022,1,202203,ANK-1,ANGKE,JL. DURI KOSAMBI CILEDUG,"-6,175882","106,728695",KIMIA,COD DICHROMAT,MG L,25,30
5,2022,2022,1,202203,ANK-1,ANGKE,JL. DURI KOSAMBI CILEDUG,"-6,175882","106,728695",KIMIA,DO,MG L,4,"1,42"
6,2022,2022,1,202203,ANK-1,ANGKE,JL. DURI KOSAMBI CILEDUG,"-6,175882","106,728695",KIMIA,NITRAT NO3,MG L,10,"0,18"
7,2022,2022,1,202203,ANK-1,ANGKE,JL. DURI KOSAMBI CILEDUG,"-6,175882","106,728695",KIMIA,NITRIT,MG L,"0,06","0,15"


In [3]:
def to_num(s):
    s = str(s).strip()
    if s in ("-", ""): return np.nan
    try: return float(s.replace(",", "."))
    except ValueError: return np.nan

K = ["periode", "titik_sampel"]
status = raw[raw.hasil_pengukuran.str.startswith("CEMAR")].drop_duplicates(K + ["hasil_pengukuran"])[K + ["hasil_pengukuran"]]
status = status.rename(columns={"hasil_pengukuran": "status_ip"})
nilai = raw[(raw.parameter == "-") & ~raw.hasil_pengukuran.str.startswith("CEMAR")].copy()
nilai["nilai_ip"] = nilai.hasil_pengukuran.map(to_num)
nilai = nilai.drop_duplicates(K + ["nilai_ip"])[K + ["nilai_ip"]]

prm = raw[raw.jenis_parameter.isin(["FISIKA", "KIMIA", "BIOLOGI"])].copy()
prm["nilai"] = prm.hasil_pengukuran.map(to_num)
n_before = len(prm)
prm = prm.drop_duplicates(K + ["parameter", "hasil_pengukuran"]).drop_duplicates(K + ["parameter"])
print(f"Baris parameter: {n_before} -> {len(prm)} setelah menghapus duplikat")
wide = prm.pivot(index=K, columns="parameter", values="nilai").reset_index()
df = wide.merge(status, on=K, how="left").merge(nilai, on=K, how="left")
print("Data format lebar:", df.shape)
df.head()

Baris parameter: 13553 -> 12974 setelah menghapus duplikat
Data format lebar: (460, 34)


,periode,titik_sampel,AMONIAK,BOD,COD DICHROMAT,CROM HEXAVALEN CR6,DO,F,FECAL COLIFORM,FENOL,...,TEMBAGA CU,TIMAH HITAM PB,TOTAL COLIFORM,TOTAL N,TOTAL P,WARNA,ZAT PADAT TERLARUT TDS,ZAT PADAT TERSUSPENSI TSS,status_ip,nilai_ip
0,1,ANK-1,3.490,15.35,30.0,0.0029,1.42,0.019,900000.0,0.0026,...,0.009,0.019,2200000.0,NaN,0.0159,60.0,170.0,19.0,CEMAR BERAT,11.249000
1,1,ANK-2,11.560,13.54,69.0,0.0029,0.71,0.019,22000000.0,0.0051,...,0.009,0.019,30000000.0,NaN,0.0159,88.0,349.0,26.0,CEMAR BERAT,16.206000
2,1,ANK-3,11.060,24.68,106.0,0.0029,1.16,0.019,17000000.0,0.0028,...,0.009,0.019,50000000.0,NaN,0.0159,106.0,351.0,47.0,CEMAR BERAT,15.826000
3,1,BLC-1,0.109,22.14,77.0,0.0029,3.90,0.330,11000000.0,0.0031,...,0.009,0.019,24000000.0,NaN,0.0159,109.0,166.3,25.0,CEMAR BERAT,15.115391
4,1,BRN-1,7.310,38.39,327.0,0.0029,1.35,0.019,13000000.0,0.0044,...,0.009,0.019,28000000.0,NaN,0.0260,299.0,290.0,160.0,CEMAR BERAT,15.446000


## 2. Data Understanding & EDA

In [4]:
PARAM = [c for c in df.columns if c not in K + ["status_ip", "nilai_ip"]]
print(f"Jumlah baris x kolom: {df.shape[0]} x {df.shape[1]}  ({len(PARAM)} parameter, 1 target)")
print("\nDistribusi target (status_ip):")
print(df.status_ip.value_counts(dropna=False))
print("\nTipe data:"); print(df[PARAM].dtypes.value_counts())

Jumlah baris x kolom: 460 x 34  (30 parameter, 1 target)

Distribusi target (status_ip):
CEMAR BERAT     324
CEMAR SEDANG    115
CEMAR RINGAN     15
NaN               6
Name: status_ip, dtype: int64

Tipe data:
float64    30
dtype: int64


In [5]:
miss = (df[PARAM].isna().mean() * 100).sort_values(ascending=False)
print("Missing value per parameter (%):"); print(miss[miss > 0].round(2).to_string())
print("\nBaris tanpa label status_ip:", df.status_ip.isna().sum())
print("Missing per periode untuk parameter yang kosong sistematis:")
print(df.groupby("periode")[["AMONIAK", "MBAS", "SIANIDA", "KLORIDA", "SULFAT"]].apply(lambda g: g.isna().mean().round(2)))

fig, ax = plt.subplots(figsize=(8, 5))
miss[miss > 0].sort_values().plot.barh(ax=ax, color="#c0504d")
ax.set_xlabel("% missing value"); ax.set_title("Missing value per parameter (data mentah)"); plt.tight_layout()
plt.savefig("fig_missing.png", dpi=130); plt.show()

Missing value per parameter (%):
TOTAL N                      100.00
MBAS                          26.74
AMONIAK                       23.70
KLORIDA                       23.70
WARNA                         23.70
SULFAT                        23.70
NI                            23.70
SIANIDA                       23.48
ZAT PADAT TERSUSPENSI TSS      0.87
ZAT PADAT TERLARUT TDS         0.87
SENG ZN                        0.65
COD DICHROMAT                  0.65
CROM HEXAVALEN CR6             0.65
TOTAL COLIFORM                 0.65
TIMAH HITAM PB                 0.65
TEMBAGA CU                     0.65
FENOL                          0.65
BOD                            0.65
MINYAK DAN LEMAK               0.65
FECAL COLIFORM                 0.65
KADMIUM CD                     0.65
HG                             0.65
PH                             0.65
NITRIT                         0.43
H2S                            0.43
NITRAT NO3                     0.43
DO                             

In [6]:
print(df[PARAM].describe().T[["count", "min", "50%", "max"]].round(3).to_string())

def iqr_bounds(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return q1 - 1.5 * i, q3 + 1.5 * i

rows = []
for c in PARAM:
    lo, hi = iqr_bounds(df[c].dropna()); n = int(((df[c] < lo) | (df[c] > hi)).sum())
    rows.append((c, n, round(100 * n / df[c].notna().sum(), 1)))
out = pd.DataFrame(rows, columns=["parameter", "n_outlier_IQR", "persen"]).sort_values("n_outlier_IQR", ascending=False)
print("\nOutlier (aturan IQR 1,5) pada data mentah:"); print(out.head(12).to_string(index=False))

                           count       min          50%           max
AMONIAK                    351.0     0.109        4.180  2.719000e+01
BOD                        457.0     1.330       11.970  6.574000e+01
COD DICHROMAT              457.0     4.300       54.000  6.100000e+02
CROM HEXAVALEN CR6         457.0     0.003        0.003  2.400000e-02
DO                         458.0     0.000        1.650  9.000000e+00
F                          458.0     0.019        0.095  1.400000e+00
FECAL COLIFORM             457.0  1700.000  2800000.000  7.000000e+07
FENOL                      457.0     0.001        0.002  2.406000e+00
H2S                        458.0     0.001        0.005  1.654000e+00
HG                         457.0     0.000        0.000  1.000000e-03
KADMIUM CD                 457.0     0.006        0.006  7.000000e-03
KLORIDA                    351.0     1.000       32.150  5.419860e+03
KLORIN BEBAS               458.0     0.030        0.300  1.280000e+00
MBAS                

In [7]:
show = ["BOD", "COD DICHROMAT", "FECAL COLIFORM", "ZAT PADAT TERSUSPENSI TSS"]
fig, axs = plt.subplots(1, 4, figsize=(13, 4))
for a, c in zip(axs, show):
    a.boxplot(df[c].dropna(), vert=True, flierprops=dict(marker="o", markersize=3, markerfacecolor="#c0504d"))
    a.set_title(c, fontsize=9); a.set_yscale("log"); a.set_xticks([])
fig.suptitle("Boxplot data mentah (sumbu log) — banyak outlier pada sisi tinggi"); plt.tight_layout()
plt.savefig("fig_boxplot_raw.png", dpi=130); plt.show()

In [8]:
order = ["CEMAR RINGAN", "CEMAR SEDANG", "CEMAR BERAT"]
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
df.status_ip.value_counts().reindex(order).plot.bar(ax=axs[0], color=["#9bbb59", "#f79646", "#c0504d"])
axs[0].set_title("Distribusi status pencemaran"); axs[0].set_xlabel(""); axs[0].tick_params(axis="x", rotation=0)
d_ = df.dropna(subset=["status_ip"])
for s_, col in zip(order, ["#9bbb59", "#f79646", "#c0504d"]):
    axs[1].hist(np.log10(d_.loc[d_.status_ip == s_, "FECAL COLIFORM"].dropna()), bins=15, alpha=.6, label=s_, color=col)
axs[1].set_xlabel("log10(Fecal Coliform, MPN/100 mL)"); axs[1].set_title("Fecal coliform per kelas"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_target_eda.png", dpi=130); plt.show()

## 3. Data Preparation

In [9]:
BAKU_MUTU = { 
    "BOD": 3, "COD DICHROMAT": 25, "ZAT PADAT TERSUSPENSI TSS": 50, "ZAT PADAT TERLARUT TDS": 1000,
    "TOTAL P": 0.2, "NITRAT NO3": 10, "NITRIT": 0.06, "FECAL COLIFORM": 1000, "TOTAL COLIFORM": 5000,
    "F": 1.5, "FENOL": 0.005, "H2S": 0.002, "HG": 0.002, "KADMIUM CD": 0.01, "KLORIN BEBAS": 0.03,
    "MINYAK DAN LEMAK": 1, "SENG ZN": 0.05, "TEMBAGA CU": 0.02, "TIMAH HITAM PB": 0.03, "CROM HEXAVALEN CR6": 0.05,
}

cek = raw[raw.parameter.isin(BAKU_MUTU)].drop_duplicates("parameter").set_index("parameter").baku_mutu.map(to_num)
assert all(abs(cek[k] - v) < 1e-9 for k, v in BAKU_MUTU.items()), "baku mutu tidak cocok"


d = df.dropna(subset=["status_ip"]).copy()
keep = [c for c in PARAM if df[c].isna().mean() <= 0.20]
dropped = [c for c in PARAM if c not in keep]
d = d.dropna(subset=keep, how="all")
print("Kolom dibuang (missing > 20%):", dropped)

FEATURES = [c for c in keep if c in BAKU_MUTU]
print(len(FEATURES), "fitur rasio:", FEATURES)
print("Sampel akhir:", len(d), "| distribusi kelas:", d.status_ip.value_counts().to_dict())

Kolom dibuang (missing > 20%): ['AMONIAK', 'KLORIDA', 'MBAS', 'NI', 'SIANIDA', 'SULFAT', 'TOTAL N', 'WARNA']
20 fitur rasio: ['BOD', 'COD DICHROMAT', 'CROM HEXAVALEN CR6', 'F', 'FECAL COLIFORM', 'FENOL', 'H2S', 'HG', 'KADMIUM CD', 'KLORIN BEBAS', 'MINYAK DAN LEMAK', 'NITRAT NO3', 'NITRIT', 'SENG ZN', 'TEMBAGA CU', 'TIMAH HITAM PB', 'TOTAL COLIFORM', 'TOTAL P', 'ZAT PADAT TERLARUT TDS', 'ZAT PADAT TERSUSPENSI TSS']
Sampel akhir: 453 | distribusi kelas: {'CEMAR BERAT': 323, 'CEMAR SEDANG': 115, 'CEMAR RINGAN': 15}


In [10]:
class RatioLog(BaseEstimator, TransformerMixin):
    """log10(nilai / baku mutu)"""
    def __init__(self, cols=None): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = np.asarray(X, float); bm = np.array([BAKU_MUTU[c] for c in self.cols])
        return np.log10(X / bm + 1e-3)

class IQRCap(BaseEstimator, TransformerMixin):
    """Winsorizing: nilai di luar [Q1-1.5IQR, Q3+1.5IQR] (dari data latih) dipotong ke batas."""
    def fit(self, X, y=None):
        q1, q3 = np.percentile(X, [25, 75], axis=0); i = q3 - q1
        self.lo_, self.hi_ = q1 - 1.5 * i, q3 + 1.5 * i; return self
    def transform(self, X): return np.clip(X, self.lo_, self.hi_)

def make_pipe(cols, k=3, cap=True):
    steps = [("impute", SimpleImputer(strategy="median")), ("ratiolog", RatioLog(cols))]
    if cap: steps.append(("iqrcap", IQRCap()))
    steps += [("minmax", MinMaxScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))]
    return Pipeline(steps)

V = [col for col in ["BOD", "FECAL COLIFORM", "TOTAL COLIFORM"] if col in d.columns]

if len(V) > 0:
    Xv = d[V]
    Zv = RatioLog(V).transform(SimpleImputer(strategy="median").fit_transform(Xv))
    Cv = IQRCap().fit(Zv).transform(Zv)
    
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    boxplot_labels = [col.split()[0].capitalize() for col in V]
    
    try:
        axs[0].boxplot([np.log10(Xv[c].dropna()) for c in V], labels=boxplot_labels)
    except TypeError:
         axs[0].boxplot([np.log10(Xv[c].dropna()) for c in V])
         axs[0].set_xticklabels(boxplot_labels)
         
    try:
        axs[1].boxplot(Zv, labels=boxplot_labels)
    except TypeError:
        axs[1].boxplot(Zv)
        axs[1].set_xticklabels(boxplot_labels)
        
    try:
        axs[2].boxplot(Cv, labels=boxplot_labels)
    except TypeError:
        axs[2].boxplot(Cv)
        axs[2].set_xticklabels(boxplot_labels)
        
    axs[0].set_title("Data mentah (log10 nilai)")
    axs[1].set_title("log10(rasio baku mutu)")
    axs[2].set_title("Setelah IQR-cap")
    
    plt.tight_layout(); plt.savefig("fig_boxplot_after.png", dpi=130); plt.show()
    print("Outlier terdeteksi (IQR, skala log):", {c: int(((Zv[:, j] < IQRCap().fit(Zv).lo_[j]) | (Zv[:, j] > IQRCap().fit(Zv).hi_[j])).sum()) for j, c in enumerate(V)})
else:
    print("Fitur visualisasi tidak ditemukan di DataFrame.")

Outlier terdeteksi (IQR, skala log): {'BOD': 5, 'FECAL COLIFORM': 0, 'TOTAL COLIFORM': 0}


## 4. Pemodelan KNN

In [11]:
y = d.status_ip
Xtr_all, Xte_all, ytr, yte = train_test_split(d[FEATURES], y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Latih:", len(Xtr_all), "| Uji:", len(Xte_all)); print(ytr.value_counts().to_dict(), yte.value_counts().to_dict())
base = yte.value_counts(normalize=True).max()
print(f"Baseline (selalu tebak kelas mayoritas): akurasi = {base:.3f}")

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
def fit_eval(cols, nama):
    gs = GridSearchCV(make_pipe(cols), {"knn__n_neighbors": [1, 3, 5, 7, 9, 11]}, cv=cv, scoring="f1_macro")
    gs.fit(Xtr_all[cols], ytr)
    pred = gs.predict(Xte_all[cols])
    print(f"\n=== {nama} ({len(cols)} fitur) | k terbaik = {gs.best_params_['knn__n_neighbors']} | CV macro-F1 = {gs.best_score_:.3f}")
    print(f"Akurasi uji = {accuracy_score(yte, pred):.3f} | macro-F1 uji = {f1_score(yte, pred, average='macro'):.3f}")
    print(classification_report(yte, pred, zero_division=0))
    return gs, pred

gsA, predA = fit_eval(["BOD", "FECAL COLIFORM", "TOTAL COLIFORM"], "Model A (final, replika manual)")
gsB, predB = fit_eval(FEATURES, "Model B (pembanding)")

Latih: 362 | Uji: 91
{'CEMAR BERAT': 258, 'CEMAR SEDANG': 92, 'CEMAR RINGAN': 12} {'CEMAR BERAT': 65, 'CEMAR SEDANG': 23, 'CEMAR RINGAN': 3}
Baseline (selalu tebak kelas mayoritas): akurasi = 0.714

=== Model A (final, replika manual) (3 fitur) | k terbaik = 9 | CV macro-F1 = 0.931
Akurasi uji = 0.934 | macro-F1 uji = 0.898
              precision    recall  f1-score   support

 CEMAR BERAT       0.98      0.94      0.96        65
CEMAR RINGAN       0.75      1.00      0.86         3
CEMAR SEDANG       0.84      0.91      0.87        23

    accuracy                           0.93        91
   macro avg       0.86      0.95      0.90        91
weighted avg       0.94      0.93      0.94        91


=== Model B (pembanding) (20 fitur) | k terbaik = 3 | CV macro-F1 = 0.790
Akurasi uji = 0.912 | macro-F1 uji = 0.729
              precision    recall  f1-score   support

 CEMAR BERAT       0.95      0.97      0.96        65
CEMAR RINGAN       0.50      0.33      0.40         3
CEMAR SEDANG

In [12]:
labels = ["CEMAR RINGAN", "CEMAR SEDANG", "CEMAR BERAT"]
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for a, p, t in zip(axs, [predA, predB], ["Model A (3 fitur, final)", "Model B (20 fitur, pembanding)"]):
    ConfusionMatrixDisplay(confusion_matrix(yte, p, labels=labels), display_labels=["Ringan", "Sedang", "Berat"]).plot(ax=a, colorbar=False, cmap="Blues")
    a.set_title(t)
plt.tight_layout(); plt.savefig("fig_confusion.png", dpi=130); plt.show()

In [13]:
from sklearn.model_selection import cross_val_score
for nama, cap in [("tanpa cap outlier", False), ("dengan IQR-cap", True)]:
    s = cross_val_score(make_pipe(FEATURES, 3, cap), d[FEATURES], y, cv=cv, scoring="f1_macro")
    print(f"{nama:20s} macro-F1 CV = {s.mean():.3f} ± {s.std():.3f}")

tanpa cap outlier    macro-F1 CV = 0.769 ± 0.117
dengan IQR-cap       macro-F1 CV = 0.743 ± 0.117


## 5. Verifikasi hitungan manual (Soal 2)

In [15]:
S = pd.read_csv("sample17_raw.csv") 
F3 = ["BOD", "FECAL COLIFORM", "TOTAL COLIFORM"]
te_ids = ["KRT-4", "KML-2"]
te, tr = S[S.titik_sampel.isin(te_ids)], S[~S.titik_sampel.isin(te_ids)]
print("Median BOD (data latih, nilai ada) =", tr.BOD.median())
pipe = make_pipe(F3, k=3).fit(tr[F3], tr.status_ip)
print("Prediksi KNN (k=3):", dict(zip(te.titik_sampel, pipe.predict(te[F3]))))
print("Label sebenarnya  :", dict(zip(te.titik_sampel, te.status_ip)))
Z = pipe[:-1].transform(te[F3]); Ztr = pipe[:-1].transform(tr[F3])
for t, z in zip(te.titik_sampel, Z):
    dist = np.sqrt(((Ztr - z) ** 2).sum(axis=1))
    print(t, "-> 3 tetangga terdekat:", [(tr.titik_sampel.iloc[i], tr.status_ip.iloc[i], round(dist[i], 4)) for i in np.argsort(dist)[:3]])

Median BOD (data latih, nilai ada) = 11.09
Prediksi KNN (k=3): {'KRT-4': 'CEMAR SEDANG', 'KML-2': 'CEMAR BERAT'}
Label sebenarnya  : {'KRT-4': 'CEMAR SEDANG', 'KML-2': 'CEMAR BERAT'}
KRT-4 -> 3 tetangga terdekat: [('PSG-3', 'CEMAR SEDANG', 0.2672), ('KLB-5', 'CEMAR SEDANG', 0.2806), ('SPK-3', 'CEMAR SEDANG', 0.2998)]
KML-2 -> 3 tetangga terdekat: [('CPN-3', 'CEMAR BERAT', 0.1304), ('CPN-7', 'CEMAR BERAT', 0.1426), ('CLW3-2', 'CEMAR BERAT', 0.1765)]


## 6. Kesimpulan
Model KNN dengan 3 fitur menggunakan rasio terhadap baku mutu dan
teknik lain mencapai akurasi 93% dan macro-F1 0,90. Penambahan fitur
menjadi 20 (Model B) menurunkan macro-F1 menjadi 0,73. Kelas *cemar
ringan* memiliki sampel yang sedikit, membuat metriknya tidak stabil. IQR-
cap tidak meningkatkan skor, dan outlier ditangani dalam preprocessing.
Model ini juga mengaproksimasi rumus IP, berguna sebagai penyaring cepat
dan alat pembelajaran, bukan pengganti perhitungan resmi.